# Test clase `osrigol.py`

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from oscrigol import oscrigol

In [2]:
# Verificación básica de comunicación con el osciloscopio usando PyVISA
import pyvisa

rm = pyvisa.ResourceManager("@py")
inst = rm.open_resource("TCPIP0::192.168.2.2::5555::SOCKET")
inst.read_termination = "\n"
inst.write_termination = "\n"
inst.timeout = 10000

print(inst.query("*IDN?"))
inst.close()

RIGOL TECHNOLOGIES,MSO2102A,DS2F214100395,00.03.06


In [3]:
scope = oscrigol("192.168.2.2")

scope.config(
    channels=(1,),
    chanBand=("OFF",),
    chanCoup=("AC",),
    chanInv=("OFF",),
    chanImp=("OMEG",),
    trigSource="CHAN1",
    trigCoup="DC",
    trigLevel=0.0,
    trigSlope="POS",
    acquisition=1,
    mdepth=14000
)

scope.initComm()

In [4]:
scope.setEdgeTrigger("CHAN1", "POS", "DC", 0.0)
scope.setChannel(1, "OFF", "AC", "OFF", "FIFTy")

print(scope.getID())

RIGOL TECHNOLOGIES,MSO2102A,DS2F214100395,00.03.06



In [5]:
scale = scope.getVertScale(1)
offset = scope.getVertOffset(1)

print("Escala vertical inicial (CH1):", scale, "V/div")
print("Offset vertical inicial (CH1):", offset, "divisiones")

Escala vertical inicial (CH1): 0.02 V/div
Offset vertical inicial (CH1): 0.0 divisiones


In [6]:
scope.setVertScale(channel=1, vScale=0.375)
print("Escala vertical (CH1) actual:", scope.getVertScale(1), "V/div")

Escala vertical (CH1) actual: 0.375 V/div


In [7]:
vmax=scope.getVMax(1)
vmin=scope.getVMin(1)
print(f"Vmax: {vmax} V, Vmin: {vmin}")

Vmax: 0.12 V, Vmin: -0.112


In [11]:
# Prueba segunda implmentación autoajuste vertical

scope.autoAdjustVertScale(
    channels=(1,),
    mode="PEAK",
    n_iter=3,
    target_divisions=7.0,
    min_divisions=6.0,
    max_divisions=7.5,
    use_scope_steps=False,
    acq_wait=0.5,
    settle_wait=0.2,
    verbose=True
)


AutoAdjust vertical - iteración 1/3
CH1: Vmin=-0.0896 V, Vmax=0.0784 V, Vpp=0.168 V, usado=8.40 div
CH1: ajuste -> scale=0.02 → 0.024 V/div, offset=0 → 0.0056 V

AutoAdjust vertical - iteración 2/3
CH1: Vmin=-0.0896 V, Vmax=0.0784 V, Vpp=0.168 V, usado=7.00 div
CH1: no requiere ajuste. scale=0.024 V/div, offset=0.0056 V


{1: True}

In [14]:
scale = scope.getVertScale(1)
offset = scope.getVertOffset(1)

print(f"Escala final CH1: {scale:.4g} V/div")
print(f"Offset final CH1: {offset:.4g} V")

Escala final CH1: 0.03344 V/div
Offset final CH1: -0.008777 V
